# 01 — Telecom canonical data and truth isolation

**Purpose:** convert the observable telecom source into a small canonical
contract and keep evaluation truth physically separate.

This notebook does only six things:

1. standardises timestamp, entity and metric names;
2. converts wide telemetry to a long canonical table;
3. preserves units and measurement kinds;
4. marks null values as `invalid` and capped FEC values as `clipped`;
5. writes separate `SPEC-CORE` and `SPEC-EVAL` directories;
6. proves that removing evaluation files does not change `SPEC-CORE`.

It does **not** create model features, select seasonal periods, calculate
anomaly scores or choose thresholds.

## 1. Setup and run controls

Put this notebook and `week1_core.py` together in
`MyDrive/anomaly_detection/research/week1/`.

Existing output directories are never overwritten. Use a new `RUN_ID`
when the source or selection changes.

In [ ]:
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    CORE_SCHEMAS,
    CORE_TABLES,
    CORE_VERSION,
    EVAL_SCHEMAS,
    EVAL_TABLES,
    Selection,
    audit_core,
    canonical_frame_hash,
    discover_telecom,
    materialise_telecom,
    source_file,
    translate_telemetry_batch,
    write_json,
)

source_candidates = [
    DRIVE_ROOT / "telco_syntetic_data",
    DRIVE_ROOT / "telco_syntetic_data" / "native",
    DRIVE_ROOT / "telco_synthetic_data",
    DRIVE_ROOT / "telco_synthetic_data" / "native",
]
default_source = next(
    (
        path for path in source_candidates
        if (path / "reference_dataset.parquet").is_file()
    ),
    source_candidates[0],
)

SOURCE = Path(os.getenv("TELECOM_SOURCE_ROOT", str(default_source)))
RUN_ID = os.getenv("TELECOM_RUN_ID", "telecom_minimal_v1")
RUN_ROOT = (
    DRIVE_ROOT / "outputs" / "research" / f"v{CORE_VERSION}"
    / "telecom" / RUN_ID
)

entity_ids = tuple(filter(
    None,
    (
        value.strip()
        for value in os.getenv("TELECOM_ENTITY_IDS", "").split(",")
    ),
))
selection = Selection(
    entity_ids=entity_ids,
    start=os.getenv("TELECOM_START") or None,
    end=os.getenv("TELECOM_END") or None,
    batch_rows=int(os.getenv("TELECOM_BATCH_ROWS", "100000")),
)

display(pd.Series({
    "source": str(SOURCE),
    "output": str(RUN_ROOT),
    "entities": entity_ids or "all",
    "start": selection.start or "first observation",
    "end": selection.end or "last observation",
    "batch_rows": selection.batch_rows,
}, name="value").to_frame())

## 2. Minimal contract

`SPEC-CORE` is the only directory available to EDA and modelling.
`SPEC-EVAL` contains fault intervals and tickets and may be unmounted.

Service windows define when an entity was expected to produce telemetry.
Collection gaps record absent timestamps only inside those windows.

In [ ]:
display(pd.DataFrame([
    {
        "directory": "SPEC-CORE",
        "table": table,
        "fields": ", ".join(fields),
    }
    for table, fields in CORE_SCHEMAS.items()
]))

display(pd.DataFrame([
    {
        "directory": "SPEC-EVAL",
        "table": table,
        "fields": ", ".join(fields),
    }
    for table, fields in EVAL_SCHEMAS.items()
]))

assert tuple(CORE_SCHEMAS) == CORE_TABLES
assert tuple(EVAL_SCHEMAS) == EVAL_TABLES

## 3. Telecom metric map

This is the Telecom Pack used by the translator. The first five columns
describe the source field. `clip_at` is populated only when the native
generator reports a capped value.

Aggregation rules, anomaly direction and feature formulas deliberately do
not belong here; they are modelling decisions.

In [ ]:
METRIC_COLUMNS = [
    "native_field", "metric_id", "entity_type",
    "measurement_kind", "unit", "clip_at",
]
metric_map = pd.DataFrame([
    ("rx_power_dbm", "rx_power_dbm", "ont", "gauge", "dBm", None),
    ("olt_rx_power_dbm", "olt_rx_power_dbm", "ont", "gauge", "dBm", None),
    ("tx_power_dbm", "tx_power_dbm", "ont", "gauge", "dBm", None),
    ("temperature_c", "temperature_c", "ont", "gauge", "degC", None),
    ("bias_current_ma", "bias_current_ma", "ont", "gauge", "mA", None),
    ("voltage_v", "voltage_v", "ont", "gauge", "V", None),
    ("ber", "ber", "ont", "bounded_fraction", "ratio", None),
    ("fec_count", "fec_count", "ont", "interval_count", "count", 5_000_000),
    ("crc_errors", "crc_errors", "ont", "interval_count", "count", None),
    ("uptime_s", "uptime_s", "ont", "cumulative_counter", "s", None),
    ("reboot_count", "reboot_count", "ont", "cumulative_counter", "count", None),
    ("throughput_mbps", "throughput_mbps", "ont", "gauge", "Mbps", None),
], columns=METRIC_COLUMNS)

relation_mappings = [
    ("olt_id", "pon_port", "contains"),
    ("pon_port", "splitter_l1", "contains"),
    ("splitter_l1", "splitter_l2", "contains"),
    ("splitter_l2", "ont_id", "serves"),
]

display(metric_map)
display(pd.DataFrame(
    relation_mappings,
    columns=["parent_field", "child_field", "relation_type"],
))

## 4. Validate the source and materialise the contract

In [ ]:
inventory = discover_telecom(SOURCE)
display(pd.Series(inventory, name="value").to_frame())
assert inventory["core_ready"], inventory
assert inventory["evaluation_ready"], inventory

panel_schema = pq.ParquetFile(
    source_file(SOURCE, "reference_dataset.parquet")
).schema_arrow.names
truth_columns = [
    column for column in panel_schema
    if str(column).startswith("gt_")
]
assert not truth_columns, truth_columns

report = materialise_telecom(
    SOURCE,
    RUN_ROOT,
    metric_map,
    relation_mappings,
    selection=selection,
    include_evaluation=True,
)

display(pd.Series(
    report["core_manifest"]["row_counts"],
    name="rows",
).to_frame())
display(pd.Series(
    report["evaluation_manifest"]["row_counts"],
    name="rows",
).to_frame())

## 5. Inspect and validate the outputs

The explicit schemas below are the contract. A future change must update
the contract version rather than silently adding columns.

In [ ]:
CORE = RUN_ROOT / "SPEC-CORE"
EVALUATION = RUN_ROOT / "SPEC-EVAL"

core_manifest = json.loads((CORE / "manifest.json").read_text())
eval_manifest = json.loads((EVALUATION / "manifest.json").read_text())
audit = audit_core(CORE)

catalogue = pd.read_parquet(CORE / "metric_catalogue.parquet")
entities = pd.read_parquet(CORE / "entity_registry.parquet")
relations = pd.read_parquet(CORE / "entity_relations.parquet")
gaps = pd.read_parquet(CORE / "collection_gaps.parquet")
telemetry_sample = pd.read_parquet(
    sorted((CORE / "telemetry").glob("part-*.parquet"))[0]
).head(10)

assert set(core_manifest["tables"]) == set(CORE_TABLES)
assert set(eval_manifest["tables"]) == set(EVAL_TABLES)
assert set(catalogue["measurement_kind"]) <= {
    "gauge", "bounded_fraction",
    "interval_count", "cumulative_counter",
}
assert entities.loc[
    entities["entity_type"].eq("ont"), "valid_from"
].notna().all()
assert set(relations["relation_type"]) <= {"contains", "serves"}

display(telemetry_sample)
display(catalogue)
display(entities.head(10))
display(relations.head(10))
display(gaps.head(10))
display(pd.Series(audit, name="value").to_frame())
print("PASS — minimal contract and physical truth separation")

### Quality-rule unit check

This small deterministic check proves the rules even when a development
slice happens not to contain a null or capped FEC value.

In [ ]:
quality_probe = pd.DataFrame({
    "timestamp_utc": pd.to_datetime([
        "2026-01-01T00:00:00Z",
        "2026-01-01T00:15:00Z",
    ]),
    "ont_id": ["ONT-TEST", "ONT-TEST"],
    "fec_count": [5_000_000, 10],
    "temperature_c": [40.0, None],
})
probe_map = metric_map.loc[
    metric_map["native_field"].isin(
        ["fec_count", "temperature_c"]
    )
]
translated_probe = translate_telemetry_batch(
    quality_probe,
    probe_map,
)
observed_codes = set(
    translated_probe["quality_code"]
)
assert {"measured", "invalid", "clipped"} <= observed_codes
display(translated_probe)

## 6. Truth-isolation test

The translator is run from two source directories:

- the original source, where evaluation files are present;
- a redacted source containing only observable inputs.

Their logical `SPEC-CORE` hashes must be identical. The negative control
then injects a fake fault field and proves that the same comparison would
fail if truth leaked.

In [ ]:
panel = pq.ParquetFile(
    source_file(SOURCE, "reference_dataset.parquet")
)
first_entities = (
    next(panel.iter_batches(
        batch_size=10_000,
        columns=["ont_id"],
    ))
    .to_pandas()["ont_id"]
    .astype(str)
    .drop_duplicates()
    .head(2)
    .tolist()
)
isolation_selection = Selection(
    entity_ids=tuple(first_entities),
    batch_rows=20_000,
)

with tempfile.TemporaryDirectory(
    prefix="telecom-truth-isolation-"
) as temporary_directory:
    temporary = Path(temporary_directory)
    redacted_source = temporary / "redacted_source"
    redacted_source.mkdir()

    for filename in [
        "reference_dataset.parquet",
        "topology.csv",
        "entity_service_windows.csv",
    ]:
        shutil.copy2(
            source_file(SOURCE, filename),
            redacted_source / filename,
        )

    original_run = temporary / "original_run"
    redacted_run = temporary / "redacted_run"
    original = materialise_telecom(
        SOURCE,
        original_run,
        metric_map,
        relation_mappings,
        selection=isolation_selection,
        include_evaluation=True,
    )
    redacted = materialise_telecom(
        redacted_source,
        redacted_run,
        metric_map,
        relation_mappings,
        selection=isolation_selection,
        include_evaluation=False,
    )

    original_hashes = original["core_manifest"][
        "canonical_content_hashes"
    ]
    redacted_hashes = redacted["core_manifest"][
        "canonical_content_hashes"
    ]
    assert original_hashes == redacted_hashes

    leaky_catalogue = pd.read_parquet(
        original_run / "SPEC-CORE" / "metric_catalogue.parquet"
    )
    leaky_catalogue["fault_id"] = "DELIBERATE-LEAK"
    leaky_hashes = dict(original_hashes)
    leaky_hashes["metric_catalogue"] = canonical_frame_hash(
        leaky_catalogue,
        sort_by=leaky_catalogue.columns,
    )
    assert leaky_hashes != redacted_hashes

isolation_result = {
    "entities": first_entities,
    "original_equals_redacted": True,
    "negative_control_detected": True,
}
display(pd.Series(isolation_result, name="value").to_frame())
print("PASS — evaluation truth cannot change SPEC-CORE")

## 7. Save the acceptance report

In [ ]:
acceptance_report = {
    "contract_version": CORE_VERSION,
    "run_root": str(RUN_ROOT),
    "core_tables": list(CORE_TABLES),
    "evaluation_tables": list(EVAL_TABLES),
    "quality_counts": audit["quality_counts"],
    "truth_isolation": isolation_result,
    "model_features_created": False,
    "crc_exposure_created": False,
}
write_json(
    RUN_ROOT / "acceptance_report.json",
    acceptance_report,
)

print("Saved:", RUN_ROOT)
print(
    "NEXT: refactor the canonical EDA notebook for contract v0.4.0. "
    "Do not run the older Notebook 00 against this output."
)